# dsfb-swarm Colab Reproduction

`dsfb-swarm` is a reproducible demonstrator for deterministic spectral residual inference on swarm interaction networks. The notebook rebuilds the Rust crate from source, reruns the benchmark suite from scratch, loads the newest timestamped output bundle, shows the paper-facing figures inline, and then assembles a notebook-side PDF plus a zip archive for download.

## Technical overview

The crate evolves a time-varying swarm graph `G(t)` with weighted adjacency `A(t)`, degree `D(t)`, and Laplacian `L(t) = D(t) - A(t)`. At every step it computes the Laplacian spectrum, especially algebraic connectivity `lambda_2(t)`, and a monitored spectral stack `lambda_2(t) .. lambda_{m+1}(t)`.

The core diagnostic loop is deterministic:

- predict `hat lambda_k(t)` from previous spectral samples,
- compute residuals `r_k(t) = lambda_k(t) - hat lambda_k(t)`,
- compute residual drift and residual slew,
- compute residual envelopes and anomaly certificates,
- optionally compare mode shapes through eigenvector residuals with sign ambiguity handling,
- attenuate interactions through trust-gated effective weights `a_ij(t) = T_ij(t) * tilde a_ij(t)`.

The benchmark suite evaluates four scenarios aligned with the paper:

1. nominal coordination,
2. gradual edge degradation,
3. adversarial agent influence,
4. communication loss and fragmentation.

The important outputs are not just plots of `lambda_2(t)`. The crate also exports residuals, drift, slew, envelopes, trust trajectories, baseline comparisons, scaling metrics, and noise-stress metrics so the paper’s structural claims can be read as empirical diagnostics rather than as labels pasted onto a generic simulation.


## Notebook flow

1. Bootstrap Python and Rust tooling.
2. Locate the DSFB repository or clone it if needed.
3. Build `crates/dsfb-swarm` from source.
4. Run the benchmark suite from scratch.
5. Locate the newest `output-dsfb-swarm/<timestamp>/` directory.
6. Load CSV and JSON artifacts.
7. Display the generated figures inline.
8. Assemble a notebook-side PDF report.
9. Create a zip archive containing the full artifact bundle.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import textwrap
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display
from matplotlib.backends.backend_pdf import PdfPages


def run_command(cmd, cwd=None):
    print("$", " ".join(str(part) for part in cmd))
    subprocess.run(cmd, check=True, cwd=cwd)


subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)

if shutil.which("rustc") is None:
    run_command(["bash", "-lc", "curl https://sh.rustup.rs -sSf | sh -s -- -y"])

cargo_bin = Path.home() / ".cargo" / "bin"
os.environ["PATH"] = f"{cargo_bin}:{os.environ['PATH']}"
run_command(["rustc", "--version"])
run_command(["cargo", "--version"])


## Locate or clone the repository

The notebook first searches the local Colab filesystem for a repository containing `crates/dsfb-swarm/Cargo.toml`. If nothing suitable is found, it clones the repository into `/content/dsfb`.

By default the notebook clones `https://github.com/infinityabundance/dsfb.git` when it cannot find a local repository. You can still override that with `DSFB_REPO_URL`, `DSFB_REPO_REF`, or `DSFB_REPO_ROOT`.


In [ ]:
REPO_ROOT_OVERRIDE = os.environ.get("DSFB_REPO_ROOT", "").strip()
DEFAULT_REPO_URL = os.environ.get("DSFB_REPO_URL", "https://github.com/infinityabundance/dsfb.git").strip()
REPO_REF = os.environ.get("DSFB_REPO_REF", "main").strip() or "main"
CLONE_DIR = Path("/content/dsfb")


def has_repo_root(path: Path) -> bool:
    return (path / "crates" / "dsfb-swarm" / "Cargo.toml").exists()


def find_repo_root() -> Path | None:
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend([Path("/content"), Path("/content/dsfb"), Path("/content/repo")])
    if Path("/content").exists():
        for child in sorted(Path("/content").iterdir()):
            candidates.append(child)
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except FileNotFoundError:
            continue
        if resolved in seen:
            continue
        seen.add(resolved)
        if has_repo_root(resolved):
            return resolved
    return None


repo_root = Path(REPO_ROOT_OVERRIDE).resolve() if REPO_ROOT_OVERRIDE else find_repo_root()
if repo_root is None:
    if CLONE_DIR.exists():
        shutil.rmtree(CLONE_DIR)
    run_command(["git", "clone", "--depth", "1", "--branch", REPO_REF, DEFAULT_REPO_URL, str(CLONE_DIR)])
    repo_root = CLONE_DIR

crate_dir = repo_root / "crates" / "dsfb-swarm"
manifest_path = crate_dir / "Cargo.toml"
output_root = crate_dir / "output-dsfb-swarm"
print("repo_root:", repo_root)
print("crate_dir:", crate_dir)
print("output_root:", output_root)
assert manifest_path.exists(), manifest_path


## Build and run from scratch

The benchmark suite below rebuilds the crate and reruns all four scenarios using the crate-local output root. That preserves the self-contained execution model of `dsfb-swarm` while still producing a fresh timestamped artifact bundle every time.


In [ ]:
run_command(["cargo", "build", "--manifest-path", str(manifest_path), "--release"])
run_command([
    "cargo",
    "run",
    "--manifest-path",
    str(manifest_path),
    "--release",
    "--",
    "benchmark",
    "--all-scenarios",
    "--sizes",
    "20,50,100",
    "--noise",
    "0.01,0.05,0.10",
    "--steps",
    "120",
    "--modes",
    "4",
    "--mode-shapes",
    "--output-root",
    str(output_root),
])


In [ ]:
def latest_run_dir(root: Path) -> Path:
    runs = sorted([path for path in root.iterdir() if path.is_dir()])
    if not runs:
        raise RuntimeError(f"No timestamped runs found under {root}")
    return runs[-1]


run_dir = latest_run_dir(output_root)
figure_dir = run_dir / "figures"
report_dir = run_dir / "report"
print("latest run_dir:", run_dir)

summary = pd.read_csv(run_dir / "scenarios_summary.csv")
benchmark = pd.read_csv(run_dir / "benchmark_summary.csv")
time_series = pd.read_csv(run_dir / "time_series.csv")
manifest = json.loads((run_dir / "manifest.json").read_text())

display(Markdown("### Scenario summary"))
display(summary)
display(Markdown("### Benchmark preview"))
display(benchmark.head(16))
display(Markdown("### Manifest keys"))
display(pd.DataFrame({"key": list(manifest.keys())}))


In [ ]:
figure_names = [
    "lambda2_timeseries.png",
    "residual_timeseries.png",
    "drift_slew.png",
    "trust_evolution.png",
    "baseline_comparison.png",
    "scaling_curves.png",
    "noise_stress_curves.png",
    "multimode_comparison.png",
    "topology_snapshots.png",
]

for name in figure_names:
    display(Markdown(f"### {name}"))
    display(Image(filename=str(figure_dir / name)))


In [ ]:
pdf_path = report_dir / "dsfb_swarm_colab_report.pdf"
zip_path = report_dir / "dsfb_swarm_artifacts.zip"

with PdfPages(pdf_path) as pdf:
    fig, ax = plt.subplots(figsize=(11, 8.5))
    ax.axis("off")
    intro = textwrap.dedent(
        f"""
        dsfb-swarm Colab report

        Run directory: {run_dir}

        This notebook rebuilt the crate, reran the benchmark suite, loaded the newest timestamped output bundle,
        and assembled this PDF from the generated artifacts. The report focuses on deterministic spectral prediction,
        residual-centered diagnosis, drift and slew monitoring, trust-gated attenuation, scaling, noise stress, and
        the difference between scalar lambda_2 monitoring and stacked multi-mode monitoring.
        """
    )
    ax.text(0.02, 0.98, intro, va="top", ha="left", fontsize=12, family="monospace")
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    for title, frame in [("Scenario summary", summary), ("Benchmark summary", benchmark.head(20))]:
        fig, ax = plt.subplots(figsize=(11, 8.5))
        ax.axis("off")
        ax.set_title(title, fontsize=16, pad=20)
        table = ax.table(
            cellText=frame.round(4).astype(str).values,
            colLabels=list(frame.columns),
            loc="center",
        )
        table.auto_set_font_size(False)
        table.set_fontsize(7)
        table.scale(1, 1.3)
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

    for name in figure_names:
        img = plt.imread(figure_dir / name)
        fig, ax = plt.subplots(figsize=(11, 8.5))
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(name, fontsize=14)
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

files_to_zip = [path for path in run_dir.rglob("*") if path.is_file() and path != zip_path]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files_to_zip:
        archive.write(path, path.relative_to(run_dir))

print("Notebook PDF:", pdf_path)
print("Artifact ZIP:", zip_path)
print("Files packaged:", len(files_to_zip))


## Artifact notes

- The Rust crate already writes its own Markdown and compact PDF report into `report/`.
- This notebook adds a second PDF assembled from the generated figures and summary tables.
- The zip archive includes the run’s CSVs, JSON files, PNG figures, Markdown report, Rust PDF report, and notebook PDF.
